In [1]:
#Step 0: Import Data

# import pandas as pd
# df = pd.read_parquet('../data/final_merged.parquet')

parquet_path = '../data/final_merged.parquet'

In [6]:
import pyarrow.parquet as pq

pq.read_schema(parquet_path).names

['mention_id',
 'term',
 'sentence',
 'start_idx',
 'end_idx',
 'sentence_start_idx',
 'sentence_end_idx',
 'adjectives_fashion',
 'character_start_idx',
 'character_end_idx',
 'gender_local_pron',
 'all_genders',
 'gender_dep_pron',
 'adjectives_char',
 'gender_booknlp',
 'inferreddate',
 'decade',
 'book_id',
 'character_id']

In [ ]:

# First pass
parquet_file = pq.ParquetFile(parquet_path)
all_adjs = set()
all_fts  = set()

for batch in parquet_file.iter_batches(batch_size=CHUNK_SIZE):
    chunk = batch.to_pandas()
    all_fts.update(chunk['term'].dropna().unique())
    for adj_arr in chunk['adjectives_char']:
        if isinstance(adj_arr, np.ndarray):
            all_adjs.update(adj for adj in adj_arr if adj is not None)

adj_enc = LabelEncoder().fit(sorted(a for a in all_adjs if a is not None))
ft_enc  = LabelEncoder().fit(sorted(f for f in all_fts if f is not None))

# Recreate before second pass
parquet_file = pq.ParquetFile(parquet_path)
matrix_sparse = csr_matrix((n_adj, n_ft), dtype=np.float32)

adj_set = set(adj_enc.classes_)
ft_set  = set(ft_enc.classes_)

for i, batch in enumerate(parquet_file.iter_batches(batch_size=CHUNK_SIZE)):
    chunk = batch.to_pandas()
    rows, cols, data = [], [], []
    for _, row in chunk.iterrows():
        if not isinstance(row['adjectives_char'], np.ndarray):
            continue
        if row['term'] not in ft_set:
            continue
        ft_idx = ft_enc.transform([row['term']])[0]
        for adj in row['adjectives_char']:
            if adj not in adj_set:
                continue
            rows.append(adj_enc.transform([adj])[0])
            cols.append(ft_idx)
            data.append(1.0)

    if rows:
        matrix_sparse += csr_matrix(
            (data, (rows, cols)),
            shape=(n_adj, n_ft),
            dtype=np.float32
        )
    print(f"Chunk {i} done, nnz: {matrix_sparse.nnz}")

In [ ]:
from sklearn.decomposition import TruncatedSVD

svd = TruncatedSVD(n_components=10, random_state=42)
coords = svd.fit_transform(matrix_sparse)

print("Variance explained:", svd.explained_variance_ratio_)

In [ ]:
#Step 2: Build PCA

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA, TruncatedSVD

X = matrix.values  # numpy array

# Optional: normalize rows (each char_adj vector) to unit length
X_normed = X / (np.linalg.norm(X, axis=1, keepdims=True) + 1e-9)

# Use TruncatedSVD if the matrix is sparse/large; PCA otherwise
n_components = 10
pca = PCA(n_components=n_components)
coords = pca.fit_transform(X_normed)  # shape: (n_char_adj, n_components)

print("Variance explained per PC:", pca.explained_variance_ratio_)

In [ ]:
#Step 3: Interpret the principle components
# --- 3a. Top loading terms per PC ---
loadings = pd.DataFrame(
    pca.components_.T,
    index=matrix.columns,       # fashion terms
    columns=[f'PC{i+1}' for i in range(n_components)]
)

def top_terms(pc, n=10):
    s = loadings[pc].sort_values(key=abs, ascending=False)
    return s.head(n)

print("Top terms on PC1:")
print(top_terms('PC1'))

# --- 3b. Correlate PC scores with gender ---
# Map each char_adj back to its character's gender
# char_adj_gender: dict mapping char_adj → gender (1=male, 0=female)
from scipy.stats import pointbiserialr

gender_labels = np.array([char_adj_gender[adj] for adj in matrix.index])

for i in range(n_components):
    r, p = pointbiserialr(gender_labels, coords[:, i])
    print(f"PC{i+1}: r={r:.3f}, p={p:.4f}")

# --- 3c. Scree plot ---
import matplotlib.pyplot as plt

plt.figure(figsize=(6, 3))
plt.bar(range(1, n_components+1), pca.explained_variance_ratio_)
plt.xlabel('Principal component')
plt.ylabel('Variance explained')
plt.title('Scree plot')
plt.tight_layout()
plt.savefig('scree.png', dpi=150)

In [ ]:
#Step 4: Scatterplot

# Project characters (not just char_adjs) into PC space
# Aggregate char_adj coords back to character level first
char_coords = {}
for adj, row in zip(matrix.index, coords):
    char = adj_to_character[adj]  # your mapping
    if char not in char_coords:
        char_coords[char] = []
    char_coords[char].append(row)

char_pc = {c: np.mean(vecs, axis=0) for c, vecs in char_coords.items()}
char_df = pd.DataFrame(char_pc).T
char_df.columns = [f'PC{i+1}' for i in range(n_components)]
char_df['gender'] = [character_gender[c] for c in char_df.index]

# Plot
fig, ax = plt.subplots(figsize=(8, 6))
colors = {'male': '#378ADD', 'female': '#D4537E'}
for gender, group in char_df.groupby('gender'):
    ax.scatter(group['PC1'], group['PC2'],
               label=gender, color=colors[gender], alpha=0.7, s=60)
    for name, row in group.iterrows():
        ax.annotate(name, (row['PC1'], row['PC2']), fontsize=7, alpha=0.6)

ax.set_xlabel('PC1')
ax.set_ylabel('PC2')
ax.legend()
ax.set_title('Characters in PC1–PC2 space, colored by gender')
plt.tight_layout()
plt.savefig('pca_scatter.png', dpi=150)